# ARoF DPD-NN - Hyperparameters optimization with Optuna

In [1]:
import sys

project_path = r"C:\Users\PC\Documents\Mestrado"
file_path_2  = project_path + r"\DPD"
file_path    = project_path + r"\DPD\Optuna_studies"
sys.path.append(project_path)
sys.path.append(file_path)
sys.path.append(file_path_2)

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.constants               import pi
from scipy.signal                  import firwin, welch
from scipy.stats                   import gaussian_kde
from tqdm.notebook                 import tqdm
from tabulate                      import tabulate

from optic_private.comm.metrics    import calcEVM
from optic_private.dsp.core        import lowPassFIR, calcACLR
from optic_private.utils           import parameters, lin2dB, dBm2W

from optic_private.torchModulation import modulateGray, demodulateGray, grayMapping
from optic_private.torchMetrics    import fastBERcalc, calcMI, signal_power, papr
from optic_private.torchDSP        import upsample, firFilter, pnorm, hilbert, decimate, finddelay
from optic_private.torchModels     import mzm, linearFiberChannel, photodiode, powerAmplifier
from optic_private.torchOFDM       import modulateOFDM, demodulateOFDM

import torch as th
from torch import nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from optic_private.torchUtils import memoryLessDataSet, MLP, train_model, test_model, slidingWindowDataSet, fitFilterNN

from DPD_utils.NN_tools import NN_training, createDatasets

import optuna
from optuna.trial import TrialState

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [3]:
def plotStudyParam(n_trials, study_N1, study_N2, study_lr, study_MSE, show = False, savefig = False, file_path = None):
    trials_array = np.arange(0, n_trials)
    
    fig, axs = plt.subplots(1, 3, figsize = (20, 5))
    
    # Study N1
    xy = np.vstack([study_N1, study_MSE])
    z   = gaussian_kde(xy)(xy)
    idx = z.argsort()
    x, y, z = study_N1[idx], study_MSE[idx], z[idx]
    
    axs[0].scatter(x, 10*np.log10(y), c = z, cmap = "jet")
    axs[0].set_xlabel(r"$N_1$")
    axs[0].set_ylabel("MSE [dB]")
    axs[0].set_xlim(10, 500)
    axs[0].grid(ls = "--", color = "k", alpha = 0.5)
    
    # Study N2
    xy = np.vstack([study_N2, study_MSE])
    z   = gaussian_kde(xy)(xy)
    idx = z.argsort()
    x, y, z = study_N2[idx], study_MSE[idx], z[idx]
    
    axs[1].scatter(x, 10*np.log10(y), c = z, cmap = "jet")
    axs[1].set_xlabel(r"$N_2$")
    axs[1].set_ylabel("MSE [dB]")
    axs[1].set_xlim(10, 500)
    axs[1].grid(ls = "--", color = "k", alpha = 0.5)
    
    # Study lr
    xy = np.vstack([study_lr, study_MSE])
    z   = gaussian_kde(xy)(xy)
    idx = z.argsort()
    x, y, z = study_lr[idx], study_MSE[idx], z[idx]
    
    axs[2].scatter(x, 10*np.log10(y), c = z, cmap = "jet")
    axs[2].set_xlabel("Learning rate")
    axs[2].set_ylabel("MSE [dB]")
    axs[2].set_xscale('log')
    axs[2].set_xlim(1e-5, 1e-2)
    
    axs[2].grid(ls = "--", color = "k", alpha = 0.5)
    
    plt.tight_layout()
    
    if savefig:
        plt.savefig(file_path)
    
    if not(show):
        plt.close()

In [4]:
font = {'size':14}
plt.rc('font', **font)
plt.rcParams["font.family"] = "serif"

### 1 - Geração do Sinal OFDM

In [5]:
device = 'cuda'
includeMemory = True

# Parâmetros de modulação
modOrder = 64
modType  = 'qam'
Rb = 5e9    
SpS = 64

# OFDM signal parameters
paramOFDM = parameters()
paramOFDM.Nfft = 1024
paramOFDM.Np   = 64
paramOFDM.G    = 32
paramOFDM.hermitSymmetry = False
paramOFDM.SpS = SpS
paramOFDM.pilotCarriers = np.linspace(0, paramOFDM.Nfft-1, paramOFDM.Np, dtype=np.int64) # Pilot subcarriers
Ns = paramOFDM.Nfft - paramOFDM.Np     # Núm. de subportadoras de informação

Rs = Rb / ( Ns/(paramOFDM.Nfft + paramOFDM.G) * np.log2(modOrder) )    # Taxa de transmissão de símbolos
Fs = Rs * SpS                                                          # Taxa de amostragem

# Geração dos bits e dos símbolos QAM
batchSize = 1000//(int(paramOFDM.Nfft/128))
bits = th.randint(2, size = (batchSize*(Ns - 1), int(np.log2(modOrder))), dtype = th.float32, device = device)

symbTx  = modulateGray(bits.view(-1,).type(th.int64), modOrder, modType)    
symbTx  = pnorm(symbTx)

pilotSymb = 0.25*(max(symbTx.real) + 1j*max(symbTx.imag))
paramOFDM.pilot = pilotSymb

In [6]:
sigTx = modulateOFDM(symbTx, paramOFDM)
sigTx = pnorm(sigTx)

t = th.arange(0, len(sigTx), dtype=th.float64, device=device)*1/Fs
t_CP   = SpS * paramOFDM.G * (1/Fs)
t_symb = SpS * (paramOFDM.Nfft + paramOFDM.G) * (1/Fs)

In [7]:
table = [['Grandeza', 'Parâmetro', 'Valor'],
         ['Taxa de amostragem', 'Fs', f'{Fs*1e-9:.2f} Gsamples/s'],
         ['Taxa de símbolos', 'Rs', f'{Rs*1e-9:.2f} Gsymb/s'],
         ['Taxa de transmissão', 'Rb', f'{Rb*1e-9:.2f} Gb/s'],
         ['Ordem da constelação', 'M', f'{modOrder}'],
         ['Tamanho da FFT', 'Nfft', f'{paramOFDM.Nfft}'],
         ['Tamanho do prefixo cíclico', 'G', f'{paramOFDM.G}'],
         ['Num. port. de informação', 'Ns', f'{Ns}'],
         ['Num. port. piloto', 'Np', f'{paramOFDM.Np}'],
         ['Espaçamento entre subportadoras', 'Delta_f', f'{(Rs/paramOFDM.Nfft)*1e-6:.2f} MHz'],
         ['Duração do símbolo OFDM', 'T_symb', f'{1e9*t_symb:.2f} ns'],
         ['Duração do prefixo cíclico', 'T_CP', f'{1e9*t_CP:.2f} ns'] ]

print(tabulate(table, headers = 'firstrow', tablefmt = 'fancy_grid'))

╒═════════════════════════════════╤═════════════╤══════════════════╕
│ Grandeza                        │ Parâmetro   │ Valor            │
╞═════════════════════════════════╪═════════════╪══════════════════╡
│ Taxa de amostragem              │ Fs          │ 58.67 Gsamples/s │
├─────────────────────────────────┼─────────────┼──────────────────┤
│ Taxa de símbolos                │ Rs          │ 0.92 Gsymb/s     │
├─────────────────────────────────┼─────────────┼──────────────────┤
│ Taxa de transmissão             │ Rb          │ 5.00 Gb/s        │
├─────────────────────────────────┼─────────────┼──────────────────┤
│ Ordem da constelação            │ M           │ 64               │
├─────────────────────────────────┼─────────────┼──────────────────┤
│ Tamanho da FFT                  │ Nfft        │ 1024             │
├─────────────────────────────────┼─────────────┼──────────────────┤
│ Tamanho do prefixo cíclico      │ G           │ 32               │
├─────────────────────────────────

In [8]:
## Parametrização dos dispositivos

# Parâmetros do MZM
paramMZM = parameters()
paramMZM.Vpi = 3
paramMZM.Vb = -paramMZM.Vpi/2
paramMZM.Pin_OF = 15

# Parâmetros do canal óptico
paramChannel = parameters()
paramChannel.L = 30
paramChannel.alpha = 0.2
paramChannel.D = 16
paramChannel.Fc = 193.1e12
paramChannel.Fs = Fs

#
paramRF = parameters()
paramRF.fc_e = 10e9
paramRF.Pin_RF = 17
paramRF.Pin_PA = 5

# Parâmetros do fotodiodo
paramPD = parameters()
paramPD.ideal = False
paramPD.B = paramRF.fc_e + 2*Rs
paramPD.Ipd_sat = 50e-3
paramPD.Fs = Fs

### 2 Canal RoF

In [9]:
# Sinal RF
sigTx_RF = th.real( sigTx * th.exp(1j*2*pi*paramRF.fc_e*t) )
sigTx_RF = np.sqrt(dBm2W(paramRF.Pin_RF))*pnorm(sigTx_RF)

# Sinal óptico
Ai     = np.sqrt(dBm2W(paramMZM.Pin_OF))*th.ones(sigTx_RF.shape, device=device)
sigTxo = mzm(Ai, sigTx_RF, paramMZM)

numtaps = 4096
hopt_tx = th.tensor(firwin(numtaps, paramRF.fc_e + 2*Rs, fs = Fs)).to(device)
sigTxo  = th.sqrt(signal_power(sigTxo))*pnorm(firFilter(sigTxo, hopt_tx))

sigRxo = linearFiberChannel(sigTxo, paramChannel)

# Sinal elétrico (fotocorrente)
I_Rx = photodiode(sigRxo, paramPD)
I_Rx = pnorm(I_Rx)
I_Rx -= I_Rx.mean()

# Sinal elétrico pós-FPF
numtaps = 4096
f1 = paramRF.fc_e - 2*Rs
f2 = paramRF.fc_e + 2*Rs
hbp_RF = th.tensor(firwin(numtaps, (f1, f2), pass_zero = 'bandpass', fs = Fs)).to(device)

I_RF = firFilter(hbp_RF, I_Rx)

# Sina elétrico pós-PA
sigRx = hilbert(I_RF)*th.exp(-1j*2*pi*paramRF.fc_e*t)
sigRx = np.sqrt(dBm2W(paramRF.Pin_PA))*pnorm(sigRx)
sigRx = powerAmplifier(sigRx)

sigRx_PA = sigRx.clone().detach()

numtaps = 4096

hlp = th.tensor(firwin(numtaps, Rs/1.5, fs = Fs)).to(device)
sigRx = firFilter(hlp, sigRx)

delay = finddelay(sigRx, sigTx)
sigRx = th.roll(sigRx, -delay)

rot = th.mean(sigTx/sigRx)
sigRx = rot/th.abs(rot)*sigRx

### 3 Preparação do conjunto de dados de treinamento

In [10]:
SpS_nn = 4

paramDec = parameters()
paramDec.SpS_in = SpS
paramDec.SpS_out = SpS_nn

hlp = th.tensor(firwin(numtaps, SpS_nn*Rs/2, fs = Fs)).to(device)

sigRef = decimate(pnorm(firFilter(hlp, sigTx)), paramDec)
sigIn  = decimate(pnorm(firFilter(hlp, sigRx_PA)), paramDec).type(th.complex64)

sigRef = pnorm(sigRef).flatten()
sigIn  = pnorm(sigIn).flatten()

delay = finddelay(sigIn, sigRef)
sigIn = th.roll(sigIn, -delay)

rot = th.mean(sigRef/sigIn)
sigIn = rot/th.abs(rot)*sigIn

### 4 - Arquitetura de Aprendizado Direto (DLA)

In [11]:
paramTrain = parameters()

N = 200_000
paramTrain.num_feat = 2
paramTrain.N1 = 16
paramTrain.N2 = 8

paramTrain.divByL = 25
paramTrain.trainTestFrac = 0.75
paramTrain.batch_size = 100
paramTrain.shuffle = False
paramTrain.includeMemory = includeMemory
paramTrain.Ntaps = 8
paramTrain.augment = False

paramTrain.device = device
paramTrain.lr = 5e-3
paramTrain.epochs = 500
paramTrain.activation = "relu"
paramTrain.pgrsBar = True
paramTrain.directLearn = False

RoFChannel_model, _, _ = NN_training(sigRef[0:N], sigIn[0:N], paramTrain)

  0%|          | 0/500 [00:00<?, ?it/s]

In [13]:
def objective_DLA(trial):
    N = 150_000
    num_feat = 5
    divByL        = 25
    trainTestFrac = 0.75
    shuffle       = False
    includeMemory = True
    Ntaps         = 8
    augment       = True
    epochs        = 100
    batch_size    = 100
    
    # Optimized param
    N1 = 400 #trial.suggest_int("N1", 5, 100)
    N2 = 100 #trial.suggest_int("N2", 5, 100)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log = True)
        
    # Generate the model
    DPD_model = MLP([num_feat*Ntaps, N1, N2, 2], activation = nn.ReLU()).to(device)
    
    loss_fn = nn.MSELoss()
    optimizer = th.optim.Adam(DPD_model.parameters(), lr = lr)

    # Prepare the datasets
    train_dataloader, test_dataloader = createDatasets(sigRef[0:N], sigIn[0:N], divByL, trainTestFrac,\
                                                       batch_size, includeMemory, Ntaps, augment=augment)
    
    trainLoss = np.zeros(epochs)
    testLoss  = np.zeros(epochs)
    
    for t in range(epochs):
        # Training
        size = len(train_dataloader.dataset)
        DPD_model.train()
        
        for batch, (X, y) in enumerate(train_dataloader):    
            # Compute prediction error        
            chInput = DPD_model(X)                
            if includeMemory:
                chInput = th.view_as_complex(chInput)            
                chOutput = th.view_as_real(fitFilterNN(chInput, RoFChannel_model, Ntaps, 1, len(chInput), predict=False))
                loss = loss_fn(chOutput, y)
            else:
                chOutput = RoFChannel_model(chInput)            
                loss = loss_fn(chOutput, X)
    
            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
            if batch % 100 == 0:
                loss, current = loss.item(), (batch + 1) * len(X)
                #logg.info(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        
        # Validation      
        num_batches = len(test_dataloader)
        DPD_model.eval()
        test_loss = 0    
        with th.no_grad():
            for X, y in test_dataloader:           
                # Compute prediction error        
                chInput = DPD_model(X)           
                
                if includeMemory:
                    chInput = th.view_as_complex(chInput)
                    chOutput = th.view_as_real(fitFilterNN(chInput, RoFChannel_model, Ntaps, 1, len(chInput)))
                    test_loss += loss_fn(chOutput, y).item()
                else:
                    chOutput = RoFChannel_model(chInput)           
                    test_loss += loss_fn(chOutput, X).item()          
                                
        test_loss /= num_batches    
        
        trainLoss[t] = loss.item()
        testLoss[t]  = test_loss
        
        if (t+1) % 100 == 0:
            for g in optimizer.param_groups:
                g['lr'] = g['lr']/1.25

        trial.report(testLoss[t], t)
        
        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return testLoss[t]

In [ ]:
n_trials  = 500
n_studies = 10

for i in range(n_studies):
    print(f"Study {i+5}:")
    study_DLA = optuna.create_study(direction = "minimize", sampler = optuna.samplers.TPESampler())
    study_DLA.optimize(objective_DLA, n_trials = n_trials, timeout = 60*60*5, show_progress_bar = True)
    
    study_N1  = np.zeros(n_trials)
    study_N2  = np.zeros(n_trials)
    study_lr  = np.zeros(n_trials)
    study_MSE = np.zeros(n_trials)
    
    for j in tqdm(range(n_trials)):
        trials_params = study_DLA.trials[j].params
        study_MSE[j]  = study_DLA.trials[j].value
        
        study_N1[j]  = trials_params["N1"]
        study_N2[j]  = trials_params["N2"]
        study_lr[j]  = trials_params["lr"]
    
    np.savetxt(file_path + f'\\DPD_NN\\DLA\\DLA_study{i+5}_N1.txt', study_N1, fmt = '%f')
    np.savetxt(file_path + f'\\DPD_NN\\DLA\\DLA_study{i+5}_N2.txt', study_N2, fmt = '%f')
    np.savetxt(file_path + f'\\DPD_NN\\DLA\\DLA_study{i+5}_lr.txt', study_lr, fmt = '%f')
    np.savetxt(file_path + f'\\DPD_NN\\DLA\\DLA_study{i+5}_MSE.txt', study_MSE, fmt = '%f')
    
    print(study_DLA.best_trial.params)

In [5]:
for i in range(4):
    study_N1  = np.loadtxt(file_path + f"\\DPD_NN\\ILA\\DLA_study{i+1}_N1.txt")
    study_N2  = np.loadtxt(file_path + f"\\DPD_NN\\ILA\\DLA_study{i+1}_N2.txt")
    study_lr  = np.loadtxt(file_path + f"\\DPD_NN\\ILA\\DLA_study{i+1}_lr.txt")
    study_MSE = np.loadtxt(file_path + f"\\DPD_NN\\ILA\\DLA_study{i+1}_MSE.txt")

    arg_min_MSE = np.argmin(study_MSE)
    print(f"Best trial - Study {i+1}: MSE = {10*np.log10(study_MSE[arg_min_MSE])} dB, N1 = {study_N1[arg_min_MSE]}, N2 = {study_N2[arg_min_MSE]}, lr = {study_lr[arg_min_MSE]}")
    
    plotStudyParam(500, study_N1, study_N2, study_lr, study_MSE, show = True, savefig = True,
                   file_path = file_path + f"\\DPD_NN\\ILA\\DLA_study_{i+1}.pdf")

Best trial - Study 1: MSE = -27.03334809738469 dB, N1 = 86.0, N2 = 117.0, lr = 0.000866
Best trial - Study 2: MSE = -26.96587929403258 dB, N1 = 424.0, N2 = 92.0, lr = 0.001015
Best trial - Study 3: MSE = -27.04872914747809 dB, N1 = 32.0, N2 = 228.0, lr = 0.001429
Best trial - Study 4: MSE = -27.004927012995125 dB, N1 = 20.0, N2 = 251.0, lr = 0.001023
